# Forecasting and scenario simulation

This notebook contains the supplied simulation engine and feature helpers directly in notebook cells; no project-specific Python modules are needed. It still requires two prepared panels and two fitted models. The gap-year construction is not contained in the four supplied notebooks. Scenario settings are embedded from the supplied configuration files; their correspondence to thesis figures is not yet verified.

**Status:** publication review candidate; a complete clean-kernel execution has not been verified. Saved outputs and execution counts have been cleared. See README.md for inputs and run order.


## Workspace and configuration


In [ ]:
from pathlib import Path
import os

# Launch Jupyter in the research-notebooks directory (or one of its subdirectories).
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "publication_manifest.json").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the research-notebooks folder.")
WORK = ROOT / "work" / "04_forecast_simulation"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("Working directory:", WORK)
# Required input files belong here; outputs are written here too.
# work/ is excluded from the publication package and from Git by default.


# 0. Load and set up

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
from datetime import datetime
import plotly.express as px

# 1. Load Panel Data (Gap Year) and Engine

### Embedded implementation: features/model_schema.py

Copied from the supplied simulation folder. Numerical logic is retained.


In [ ]:
# ============================================================
# MODEL INPUT SCHEMA (Single Source of Truth)
# ============================================================

# ----- COST ANN MODEL -----

COST_NUMERIC_FEATURES = [
    "ln_gdp_o",
    "ln_gdp_d",
    "ln_dist",
    "conn_ij",
    "ln_bdi",
    "ln_fbx",
    "ln_dist_x_bulk",
    "conn_x_bulk",
    "ln_bdi_x_bulk",
    "bulk_share"
]

COST_CATEGORICAL_FEATURES = [
    "origin_isocode",
    "destination_isocode",
    "product_code"
]


# ----- TRADE ANN MODEL -----

TRADE_NUMERIC_FEATURES = [
    "Log_Origin_GDP",
    "Log_Destination_GDP",
    "cepii_contig",
    "cepii_comlang_ethno",
    "cepii_wto_o",
    "cepii_wto_d",
    "cepii_fta_wto",
    "Log_cost_per_ton_seamodes",
    "lsbci",
    "Log_origin_production_qty",
    "Log_destination_production_qty",
    "Log_Origin_Daily_Calorific_Food",
    "Log_Destination_Daily_Calorific_Food"
]

TRADE_CATEGORICAL_FEATURES = [
    "country_pair_id",
    "product_code"
]


### Embedded implementation: features/build_features.py

Copied from the supplied simulation folder. Numerical logic is retained.


In [ ]:
import numpy as np
import pandas as pd
def _zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

def _safe_log(x, eps=1e-9):
    return np.log(np.clip(x, eps, None))


def build_features(
    df: pd.DataFrame,
    *,
    col_fbx="fbx_new",
    col_bdi="bdi",
    col_gdp_o="origin_gdp_usd_current",
    col_gdp_d="destination_gdp_usd_current",
    col_dist="cepii_dist",
    col_conn="lsbci",
    col_o="origin_isocode",
    col_d="destination_isocode",
    col_year="year",
    col_prod="product_code",
    col_tons = 'tons_seamode',
    hs4_bulk_list=("1001","1002","1003","1004","1005","1006","1007","1008","1009"),
    verbose=True
):
    """
    Build model-ready features for the ANN seamode transport cost model.
    Produces:
      - log features
      - interactions
      - bulk_share probability
      - connectivity z-scores
      - minimal cleaned dataframe for ANN input
    """

    df = df.copy()

    # --------------------------------------------------
    # 0) Pair ID
    # --------------------------------------------------
    if col_o in df.columns and col_d in df.columns:
        df["pair_id"] = df[col_o].astype(str) + "|" + df[col_d].astype(str)
    else:
        raise ValueError("Missing origin/destination isocode columns")

    # --------------------------------------------------
    # 2) Log transformations
    # --------------------------------------------------
    for src, dst in [
        (col_gdp_o, "ln_gdp_o"),
        (col_gdp_d, "ln_gdp_d"),
        (col_dist, "ln_dist"),
        (col_bdi, "ln_bdi"),
        (col_fbx,"ln_fbx")
    ]:
        df[dst] = _safe_log(df[src].astype(float))


    # --------------------------------------------------
    # 3) Connectivity (z-score)
    # --------------------------------------------------
    df["conn_ij"] = _zscore(df[col_conn].astype(float))
    df["comp_ij"] = df["conn_ij"]

    # --------------------------------------------------
    # 4) Bulk-share probabilistic transformation
    # --------------------------------------------------
    df["bulk_potential"] = df[col_prod].astype(str).isin(hs4_bulk_list).astype(int)

    a, b = 6.0, 10.0  # log thresholds

    if col_tons not in df.columns:
        df["bulk_share"] = df["bulk_potential"].astype(float)
    else:
        vol = df[col_tons].fillna(0).astype(float)
        logv = np.log1p(vol)
        s = (logv - a) / (b - a)
        df["bulk_share"] = df["bulk_potential"] * s.clip(0, 1)

    # --------------------------------------------------
    # 5) Interactions
    # --------------------------------------------------
    df["ln_dist_x_bulk"] = df["ln_dist"] * df["bulk_share"]
    df["conn_x_bulk"]    = df["conn_ij"] * df["bulk_share"]
    df["ln_bdi_x_bulk"]  = df["ln_bdi"] * df["bulk_share"]

    # --------------------------------------------------
    # 6) Final cleaned dataframe
    # --------------------------------------------------
    keep_cols = [
        col_o, col_d, "pair_id", col_year, col_prod,
        "ln_dist", "conn_ij", "comp_ij",
        "ln_gdp_o", "ln_gdp_d",
        "ln_fbx", "ln_bdi", "ln_bdi_x_bulk",
        "bulk_share", "ln_dist_x_bulk", "conn_x_bulk"
    ]

    df_model = df[keep_cols].dropna().copy()

    # --------------------------------------------------
    # 7) Optional diagnostics
    # --------------------------------------------------
    if verbose and "bulk_share" in df_model.columns:
        mean_bulk = df_model["bulk_share"].mean()
        print(f"\n📊 Bulk-share Diagnostics:")
        print(f"  • Mean bulk_share: {mean_bulk:.3f}")
        print(f"  • Rows kept: {len(df_model):,}")

    return df_model


### Embedded implementation: features/ann_feature_builder.py

Copied from the supplied simulation folder. Numerical logic is retained.


In [ ]:
import pandas as pd


def make_ann_features_for_forecast(
    df_raw: pd.DataFrame,
    build_features_func,
    feature_ann_cols: list,
    categorical_ann_cols: list,
    dummy_cost_col: str = "transport_expend_per_ton_seamodes"
) -> pd.DataFrame:
    """
    Build ANN features for forward forecast data.

    This function guarantees:
    - Same feature construction logic as training
    - No hidden global dependencies
    - Safe handling of missing columns

    Parameters
    ----------
    df_raw : pd.DataFrame
        Forward baseline/stress dataframe.

    build_features_func : callable
        The original build_features function used in training.

    feature_ann_cols : list
        Numerical ANN feature columns expected by the cost model.

    categorical_ann_cols : list
        Categorical ANN feature columns expected by the cost model.

    dummy_cost_col : str
        Temporary placeholder column required by build_features.

    Returns
    -------
    pd.DataFrame
        Feature-engineered dataframe ready for ANN prediction.
    """

    df = df_raw.copy()

    # Ensure dummy column exists (needed by original build_features logic)
    if dummy_cost_col not in df.columns:
        df[dummy_cost_col] = 1.0

    # Recreate training-time feature logic
    df_feat = build_features_func(
        df,
        col_fbx="fbx_new",
        col_bdi="bdi",
        col_gdp_o="origin_gdp_usd_current",
        col_gdp_d="destination_gdp_usd_current",
        col_dist="cepii_dist",
        col_conn="lsbci",
        col_o="origin_isocode",
        col_d="destination_isocode",
        col_year="year",
        col_prod="product_code",
        verbose=False,
    )

    # Ensure required ANN inputs exist
    required_cols = feature_ann_cols + categorical_ann_cols

    missing = [c for c in required_cols if c not in df_feat.columns]
    if missing:
        raise ValueError(f"Missing ANN feature columns: {missing}")

    # Drop rows where ANN inputs are incomplete
    df_feat = df_feat.dropna(subset=required_cols).copy()

    return df_feat


### Embedded implementation: scenario_engine.py

Copied from the supplied simulation folder. Numerical logic is retained.


In [ ]:
# ============================================================
# scenario_engine.py
# Forward Baseline + Scenario Stress Simulation Engine
# ============================================================

"""
This module implements the forward-looking simulation engine
used in Chapter 5 of the thesis.

The engine performs:

1. Baseline structural projection (2025–2030)
2. Scenario-based stress injection
3. Transport cost prediction (ANN)
4. Trade volume prediction (ANN)
5. Baseline vs Stress comparison

No execution occurs automatically.
This file only defines reusable functions.
"""

# ============================================================
# IMPORTS
# ============================================================

import json
import numpy as np
import pandas as pd
from datetime import datetime


# ============================================================
# GLOBAL CONFIGURATION
# ============================================================

BASE_YEAR = 2024
FUTURE_YEARS = list(range(2025, 2031))

baseline_config = {
    "gdp_growth": 0.016,
    "population_growth": 0.013,
    "production_growth": 0.00,
    "calorific_growth": 0.00,
    "lsbci_growth": 0.00,
    "bdi_growth": 0.00,
    "fbx_growth": 0.00
}


# ============================================================
# COLUMN TAXONOMY
# ============================================================

ID_COLS = ["year", "origin_isocode", "destination_isocode", "product_code"]

LABEL_COLS = ["origin_label", "destination_label", "product_label"]

STRUCTURAL_COLS = [
    "origin_gdp_usd_current",
    "destination_gdp_usd_current",
    "origin_population_total",
    "destination_population_total",
    "cepii_dist",
    "cepii_contig",
    "cepii_comlang_ethno",
    "cepii_wto_o",
    "cepii_wto_d",
    "cepii_fta_wto",
    "origin_production_qty",
    "destination_production_qty",
    "Origin_Daily_Calorific_Food",
    "Destination_Daily_Calorific_Food",
    "lsbci",
]

COST_EXOGENOUS_COLS = [
    "fbx_new",
    "bdi",
]

BASE_FEATURE_COLS = (
    ID_COLS + LABEL_COLS + STRUCTURAL_COLS + COST_EXOGENOUS_COLS
)

ALLOW_MISSING_LABELS = True


# ============================================================
# SAFE LOG
# ============================================================

def _scenario_safe_log(x, eps=1e-9):
    x = np.asarray(x, dtype=float)
    x = np.where(x <= eps, eps, x)
    return np.log(x)


# ============================================================
# SCHEMA VALIDATION
# ============================================================

def validate_input_schema(df: pd.DataFrame):
    required = ID_COLS + STRUCTURAL_COLS + COST_EXOGENOUS_COLS
    if not ALLOW_MISSING_LABELS:
        required += LABEL_COLS

    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if not df.columns.is_unique:
        dupes = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"Duplicate column labels: {dupes}")

# ============================================================
# CORRIDOR SELECTION ENGINE
# ============================================================

def select_affected_corridors(
    df_baseline: pd.DataFrame,
    eligibility_rule,
    product_codes,
    share_threshold=0.85,
    start_year=2025,
    end_year=2030,
    min_corridors=1
):
    """
    Dynamically select affected corridors based on baseline trade volumes.

    Parameters
    ----------
    df_baseline : pd.DataFrame
        Baseline prediction output (must contain 'tons_seamode_pred').

    eligibility_rule : callable
        Function that takes df and returns boolean mask.

    product_codes : list
        HS codes to restrict corridor selection.

    share_threshold : float
        Cumulative share threshold for selecting top corridors.

    start_year, end_year : int
        Time window for baseline aggregation.

    min_corridors : int
        Ensure at least this many corridors selected.

    Returns
    -------
    List of (origin, destination) tuples.
    """

    df = df_baseline.copy()

    # Restrict to time horizon
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]

    # Apply eligibility rule
    df = df[eligibility_rule(df)]

    # Restrict to products
    if product_codes:
        df = df[df["product_code"].isin(product_codes)]

    if df.empty:
        raise ValueError("No corridors match eligibility rule.")

    # Aggregate baseline predicted tons
    corr = (
        df.groupby(["origin_isocode", "destination_isocode"], as_index=False)
          .agg(baseline_tons=("tons_seamode_pred", "sum"))
          .sort_values("baseline_tons", ascending=False)
    )

    # Compute shares
    corr["share"] = corr["baseline_tons"] / corr["baseline_tons"].sum()
    corr["cum_share"] = corr["share"].cumsum()

    # Select corridors up to threshold
    selected = corr[corr["cum_share"] <= share_threshold]

    if selected.shape[0] < min_corridors:
        selected = corr.head(min_corridors)

    return list(
        zip(selected["origin_isocode"], selected["destination_isocode"])
    )

# ============================================================
# BASELINE FEATURE CONSTRUCTION
# ============================================================

def build_future_features(
    df_fwd_forecast_gapyear: pd.DataFrame,
    base_year: int,
    future_years: list,
    baseline_config: dict
) -> pd.DataFrame:

    validate_input_schema(df_fwd_forecast_gapyear)

    df_base = df_fwd_forecast_gapyear[
        df_fwd_forecast_gapyear["year"] == base_year
    ].copy()

    keep_cols = [c for c in BASE_FEATURE_COLS if c in df_base.columns]
    df_base = df_base[keep_cols].copy()

    df_future = pd.concat(
        [df_base.assign(year=y, timeframe="future") for y in future_years],
        ignore_index=True
    )

    year_offset = (df_future["year"] - base_year).astype(int).values

    growth_map = {
        "origin_gdp_usd_current": baseline_config["gdp_growth"],
        "destination_gdp_usd_current": baseline_config["gdp_growth"],
        "origin_population_total": baseline_config["population_growth"],
        "destination_population_total": baseline_config["population_growth"],
        "origin_production_qty": baseline_config["production_growth"],
        "destination_production_qty": baseline_config["production_growth"],
        "Origin_Daily_Calorific_Food": baseline_config["calorific_growth"],
        "Destination_Daily_Calorific_Food": baseline_config["calorific_growth"],
        "lsbci": baseline_config["lsbci_growth"],
        "bdi": baseline_config["bdi_growth"],
        "fbx_new": baseline_config["fbx_growth"],
    }

    for col, rate in growth_map.items():
        if col in df_future.columns:
            base_vals = pd.to_numeric(df_future[col], errors="coerce").fillna(0.0).values
            df_future[col] = base_vals * np.power((1.0 + rate), year_offset)

    return df_future


# ============================================================
# STRESS APPLICATION
# ============================================================

def apply_combined_system_stress(
    df_future: pd.DataFrame,
    scenario_config: dict
) -> pd.DataFrame:

    df = df_future.copy()

    corridors = set(map(tuple, scenario_config["affected_corridors"]))
    products = scenario_config.get("affected_product_codes", [])

    corr_mask = pd.Series(
        list(zip(df["origin_isocode"], df["destination_isocode"])),
        index=df.index
    ).isin(corridors)

    prod_mask = (
        df["product_code"].isin(products)
        if products else pd.Series(True, index=df.index)
    )

    ts = scenario_config.get("transport_shocks", {})

    if ts.get("freight_proxy_fbx"):
        df.loc[corr_mask, "fbx_new"] *= (1 + ts["freight_proxy_fbx"])

    if ts.get("bdi_proxy"):
        df.loc[corr_mask, "bdi"] *= (1 + ts["bdi_proxy"])

    if ts.get("distance_proxy"):
        df.loc[corr_mask, "cepii_dist"] *= (1 + ts["distance_proxy"])

    if ts.get("connectivity_lsbci"):
        df.loc[corr_mask, "lsbci"] *= (1 + ts["connectivity_lsbci"])

    cs = scenario_config.get("country_shocks", {})

    for iso, shock in cs.get("production_by_origin_country", {}).items():
        m = (df["origin_isocode"] == iso) & prod_mask
        df.loc[m, "origin_production_qty"] = np.maximum(
            df.loc[m, "origin_production_qty"] * (1 + shock),
            1e-6
        )

    for iso, shock in cs.get("demand_by_destination_country", {}).items():
        m = (df["destination_isocode"] == iso) & prod_mask

        if scenario_config.get("apply_demand_to_calorific_proxy", False):
            df.loc[m, "Destination_Daily_Calorific_Food"] = np.maximum(
                df.loc[m, "Destination_Daily_Calorific_Food"] * (1 + shock),
                1e-6
            )
        else:
            df.loc[m, "destination_production_qty"] = np.maximum(
                df.loc[m, "destination_production_qty"] * (1 + shock),
                1e-6
            )

    return df


# ============================================================
# TRANSPORT COST PREDICTION
# ============================================================

def predict_transport_cost(
    df_future,
    make_ann_features_for_forecast,
    loaded_model_cost_ann,
    feature_ann_cols,
    categorical_ann_cols
):

    df = df_future.copy()

    df_ann = make_ann_features_for_forecast(df)

    X = df_ann[feature_ann_cols + categorical_ann_cols]
    ln_cost = loaded_model_cost_ann.predict(X)

    df_ann["transport_cost_per_ton_pred"] = np.exp(ln_cost)

    key_cols = ["origin_isocode", "destination_isocode", "product_code", "year"]

    df_out = df.merge(
        df_ann[key_cols + ["transport_cost_per_ton_pred"]],
        on=key_cols,
        how="left"
    )

    df_out["transport_cost_per_ton_pred"] = df_out[
        "transport_cost_per_ton_pred"
    ].fillna(0.0)

    return df_out


# ============================================================
# TRADE VOLUME PREDICTION
# ============================================================

def predict_trade_volume(
    df_with_cost,
    numerical_features,
    trade_categorical_features,
    trade_ann_pipeline
):

    df = df_with_cost.copy()

    df["year"] = df["year"].astype(int)
    df["country_pair_id"] = (
        df["origin_isocode"].astype(str) + "_" +
        df["destination_isocode"].astype(str)
    )

    df["Log_cost_per_ton_seamodes"] = _scenario_safe_log(df["transport_cost_per_ton_pred"])
    df["Log_origin_production_qty"] = _scenario_safe_log(df["origin_production_qty"])
    df["Log_destination_production_qty"] = _scenario_safe_log(df["destination_production_qty"])
    df["Log_Origin_GDP"] = _scenario_safe_log(df["origin_gdp_usd_current"])
    df["Log_Destination_GDP"] = _scenario_safe_log(df["destination_gdp_usd_current"])
    df["Log_Origin_Daily_Calorific_Food"] = _scenario_safe_log(df["Origin_Daily_Calorific_Food"])
    df["Log_Destination_Daily_Calorific_Food"] = _scenario_safe_log(df["Destination_Daily_Calorific_Food"])

    X = df[numerical_features + trade_categorical_features].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    X = X.fillna(0.0)

    df["log_tons_seamode_pred"] = trade_ann_pipeline.predict(X)
    df["tons_seamode_pred"] = np.exp(df["log_tons_seamode_pred"])

    return df


# ============================================================
# MASTER RUN FUNCTION
# ============================================================

def run_baseline_and_scenario(
    df_fwd_forecast_gapyear,
    scenario_config,
    make_ann_features_for_forecast,
    loaded_model_cost_ann,
    feature_ann_cols,
    categorical_ann_cols,
    numerical_features,
    trade_categorical_features,
    trade_ann_pipeline,
    baseline_config,
    base_year,
    future_years,
    output_dir="."
):

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    scen = scenario_config["scenario_name"].replace(" ", "_")

    df_future_base = build_future_features(
        df_fwd_forecast_gapyear,
        base_year,
        future_years,
        baseline_config
    )

    df_base_cost = predict_transport_cost(
        df_future_base,
        make_ann_features_for_forecast,
        loaded_model_cost_ann,
        feature_ann_cols,
        categorical_ann_cols
    )

    df_base_full = predict_trade_volume(
        df_base_cost,
        numerical_features,
        trade_categorical_features,
        trade_ann_pipeline
    )

    df_stress_input = apply_combined_system_stress(
        df_future_base,
        scenario_config
    )

    df_stress_cost = predict_transport_cost(
        df_stress_input,
        make_ann_features_for_forecast,
        loaded_model_cost_ann,
        feature_ann_cols,
        categorical_ann_cols
    )

    df_stress_full = predict_trade_volume(
        df_stress_cost,
        numerical_features,
        trade_categorical_features,
        trade_ann_pipeline
    )

    key_cols = ["origin_isocode", "destination_isocode", "product_code", "year"]

    df_cmp = df_base_full[key_cols + [
        "transport_cost_per_ton_pred",
        "tons_seamode_pred"
    ]].merge(
        df_stress_full[key_cols + [
            "transport_cost_per_ton_pred",
            "tons_seamode_pred"
        ]],
        on=key_cols,
        suffixes=("_baseline", "_stress"),
        how="inner"
    )

    df_cmp["pct_cost_change"] = (
        df_cmp["transport_cost_per_ton_pred_stress"] /
        df_cmp["transport_cost_per_ton_pred_baseline"] - 1
    ) * 100

    df_cmp["pct_trade_change"] = (
        df_cmp["tons_seamode_pred_stress"] /
        df_cmp["tons_seamode_pred_baseline"] - 1
    ) * 100


    return {
        "df_baseline": df_base_full,
        "df_stress": df_stress_full,
        "df_compare": df_cmp,
        "config": {
            "baseline_config": baseline_config,
            "scenario_config": scenario_config
        }
    }


**Publication review note.** Training uses volume-based bulk share; forward construction drops the volume column and falls back to a product flag. Connectivity is standardized on each prediction batch. These choices require review for consistency with training.


In [ ]:
required_files = ["data/df_fwd_forecast_gapyear.parquet", "data/df_bltr_multivar.parquet", "models/ann_model_pipeline.pkl", "models/trade_ann_log_model.pkl"]
missing_files = [name for name in required_files if not Path(name).is_file()]
if missing_files:
    raise FileNotFoundError("Required local inputs: " + ", ".join(missing_files))

df_fwd_forecast_gapyear = pd.read_parquet(
    "data/df_fwd_forecast_gapyear.parquet"
)
df_bltr_multivar = pd.read_parquet("data/df_bltr_multivar.parquet")
loaded_model_cost_ann = joblib.load("models/ann_model_pipeline.pkl")
trade_ann_pipeline = joblib.load("models/trade_ann_log_model.pkl")

print("Panel shape:", df_fwd_forecast_gapyear.shape)

def build_ann_cost_wrapper(build_features):
    def wrapper(df):
        return make_ann_features_for_forecast(
            df_raw=df,
            build_features_func=build_features,
            feature_ann_cols=COST_NUMERIC_FEATURES,
            categorical_ann_cols=COST_CATEGORICAL_FEATURES
        )
    return wrapper
ann_feature_wrapper = build_ann_cost_wrapper(build_features)
model_kwargs = dict(
    make_ann_features_for_forecast=ann_feature_wrapper,
    loaded_model_cost_ann=loaded_model_cost_ann,
    feature_ann_cols=COST_NUMERIC_FEATURES,
    categorical_ann_cols=COST_CATEGORICAL_FEATURES,
    numerical_features=TRADE_NUMERIC_FEATURES,
    trade_categorical_features=TRADE_CATEGORICAL_FEATURES,
    trade_ann_pipeline=trade_ann_pipeline
)

# 2. SCENARIO CONFIGURATION

In [ ]:
baseline_cfg = {'base_year': 2024, 'future_years': [2025, 2026, 2027, 2028, 2029, 2030], 'baseline_growth': {'gdp_growth': 0.016, 'population_growth': 0.013, 'production_growth': 0.0, 'calorific_growth': 0.0, 'lsbci_growth': 0.0, 'bdi_growth': 0.0, 'fbx_growth': 0.0}}
BASE_YEAR = baseline_cfg["base_year"]
FUTURE_YEARS = baseline_cfg["future_years"]
baseline_growth = baseline_cfg["baseline_growth"]
print(baseline_cfg)

For stress/shock test can load pre-saved configuration/case

**Publication review note.** Only the two supplied configuration files are embedded. Black Sea currently selects two corridors. No El Niño configuration was found in configs/. Transport shocks apply to all products on selected corridors; country shocks use the product filter. These settings are not asserted to reproduce the thesis figures.


In [ ]:
import copy
SCENARIOS = {'black_sea': {'scenario_name': 'Black Sea Conflict', 'affected_corridors': [['UKR', 'TUR'], ['RUS', 'EGY']], 'affected_product_codes': ['1001', '1005'], 'transport_shocks': {'freight_proxy_fbx': 0.3, 'connectivity_lsbci': -0.08, 'distance_proxy': 0.12, 'bdi_proxy': 0.0}, 'country_shocks': {'production_by_origin_country': {'UKR': -0.15, 'RUS': -0.1}, 'demand_by_destination_country': {'EGY': -0.08}}, 'apply_demand_to_calorific_proxy': False}, 'eu_ets': {'scenario_name': 'EU ETS Maritime Regulation', 'affected_corridors': [['UKR', 'ESP'], ['POL', 'DEU'], ['BRA', 'ESP'], ['BGR', 'ESP'], ['ROU', 'ESP'], ['UKR', 'PRT'], ['LTU', 'TUR'], ['NLD', 'DEU'], ['BRA', 'PRT'], ['FRA', 'PRT'], ['CAN', 'PRT'], ['HUN', 'ROU'], ['FRA', 'TUR'], ['BRA', 'NLD'], ['USA', 'ESP'], ['FRA', 'ESP'], ['POL', 'ESP'], ['GRC', 'TUR'], ['ROU', 'TUR'], ['GBR', 'ESP'], ['LTU', 'ESP'], ['CAN', 'IRL'], ['FRA', 'BEL'], ['LVA', 'ESP'], ['ROU', 'VNM'], ['DNK', 'ESP'], ['FRA', 'DEU'], ['USA', 'ITA'], ['SRB', 'ESP'], ['UKR', 'DEU'], ['ITA', 'TUR'], ['ROU', 'JOR'], ['DEU', 'ESP'], ['BGR', 'PRT'], ['BGR', 'GRC'], ['ROU', 'SDN'], ['LVA', 'TUR'], ['CAN', 'ESP'], ['ROU', 'GRC'], ['ROU', 'FRA'], ['UKR', 'ITA'], ['USA', 'PRT'], ['USA', 'NLD'], ['ROU', 'EGY'], ['DEU', 'JPN'], ['IRL', 'GBR'], ['BGR', 'TUR'], ['RUS', 'ESP'], ['SWE', 'ESP'], ['ROU', 'PRT'], ['ROU', 'GBR'], ['SWE', 'DEU'], ['NLD', 'ESP'], ['FRA', 'MUS'], ['SRB', 'PRT'], ['RUS', 'GRC'], ['ITA', 'PRT'], ['ROU', 'DEU'], ['DEU', 'TUR'], ['ESP', 'TUR'], ['HUN', 'ITA'], ['POL', 'TUR'], ['DEU', 'PRT'], ['UKR', 'NLD'], ['NLD', 'BEL'], ['UKR', 'GRC'], ['ROU', 'NLD'], ['FRA', 'GBR'], ['UKR', 'IRL'], ['FRA', 'USA'], ['BGR', 'ROU'], ['FRA', 'JPN'], ['EST', 'TUR'], ['DEU', 'NGA'], ['FRA', 'ITA'], ['FRA', 'NLD'], ['FRA', 'ROU'], ['FRA', 'MYS'], ['NLD', 'IRN'], ['POL', 'PRT'], ['DEU', 'MAR'], ['ROU', 'LBY'], ['ROU', 'LKA'], ['ARG', 'DEU'], ['ROU', 'ITA'], ['ROU', 'IRN'], ['USA', 'IRL'], ['AUT', 'ITA'], ['DNK', 'DEU'], ['RUS', 'CYP'], ['CZE', 'TUR'], ['DEU', 'NLD'], ['HRV', 'ROU'], ['FRA', 'CIV'], ['BGR', 'FRA'], ['BRA', 'ITA'], ['ESP', 'PRT'], ['ARG', 'ESP'], ['FRA', 'MAR'], ['ARG', 'POL'], ['GRC', 'ESP'], ['RUS', 'ITA']], 'affected_product_codes': [], 'transport_shocks': {'freight_proxy_fbx': 0.12, 'connectivity_lsbci': 0.0, 'distance_proxy': 0.0, 'bdi_proxy': 0.0}, 'country_shocks': {'production_by_origin_country': {}, 'demand_by_destination_country': {}}, 'apply_demand_to_calorific_proxy': False}}
SELECTED_SCENARIO = "eu_ets"
scenario_config = copy.deepcopy(SCENARIOS[SELECTED_SCENARIO])
loaded_scenario_name = scenario_config["scenario_name"]
print("Selected scenario:", loaded_scenario_name)

# Run simulation

In [ ]:
results = run_baseline_and_scenario(
    df_fwd_forecast_gapyear=df_fwd_forecast_gapyear,
    scenario_config=scenario_config,

    # ---- COST MODEL ----
    make_ann_features_for_forecast=ann_feature_wrapper,
    loaded_model_cost_ann=loaded_model_cost_ann,
    feature_ann_cols=COST_NUMERIC_FEATURES,
    categorical_ann_cols=COST_CATEGORICAL_FEATURES,

    # ---- TRADE MODEL ----
    numerical_features=TRADE_NUMERIC_FEATURES,
    trade_categorical_features=TRADE_CATEGORICAL_FEATURES,
    trade_ann_pipeline=trade_ann_pipeline,

    # ---- BASELINE CONFIG ----
    baseline_config=baseline_growth,
    base_year=BASE_YEAR,
    future_years=FUTURE_YEARS,

    output_dir="results"
)
# -------------------------------------------
# Merge FOB reference into compare dataset
# -------------------------------------------

df_fob_ref = (
    df_bltr_multivar
    .loc[
        df_bltr_multivar["fob_per_ton_seamodes"].notna(),
        ["origin_isocode", "destination_isocode", "product_code", "fob_per_ton_seamodes"]
    ]
    .groupby(
        ["origin_isocode", "destination_isocode", "product_code"],
        as_index=False
    )
    .agg(fob_per_ton_ref=("fob_per_ton_seamodes", "median"))
)

results["df_compare"] = (
    results["df_compare"]
    .merge(
        df_fob_ref,
        on=["origin_isocode", "destination_isocode", "product_code"],
        how="left"
    )
)
df_baseline = results["df_baseline"]
df_stress = results["df_stress"]
df_compare = results["df_compare"]

print(f"Simulation scenario {loaded_scenario_name} complete.")

Run this to save the results.

In [ ]:
from pathlib import Path
import json

def save_scenario_results(results, scenario_name, output_root="results"):
    base = Path(output_root) / scenario_name
    base.mkdir(parents=True, exist_ok=True)

    # Save data
    results["df_baseline"].to_parquet(base / "baseline.parquet", index=False)
    results["df_stress"].to_parquet(base / "scenario.parquet", index=False)
    results["df_compare"].to_parquet(base / "compare.parquet", index=False)

    # Save config
    with open(base / "config.json", "w") as f:
        json.dump({
            "baseline_config": baseline_growth,
            "scenario_config": scenario_config
        }, f, indent=2)

    print(f"Saved to {base}")
from pathlib import Path
import json

def export_for_dashboard(results, scenario_slug, root="results"):
    out_dir = Path(root) / scenario_slug
    out_dir.mkdir(parents=True, exist_ok=True)

    results["df_baseline"].to_json(
        out_dir / "baseline.json",
        orient="records"
    )

    results["df_stress"].to_json(
        out_dir / "scenario.json",
        orient="records"
    )

    results["df_compare"].to_json(
        out_dir / "compare.json",
        orient="records"
    )

    with open(out_dir / "config.json", "w") as f:
        json.dump(results["config"], f, indent=2)


In [ ]:
df_compare

In [ ]:
save_scenario_results(results, scenario_name=scenario_config["scenario_name"])
export_for_dashboard(results, scenario_config["scenario_name"])

In [ ]:
df_fwd_forecast_gapyear.to_json(
        "gapyear.json",
        orient="records"
    )
df_fob_ref.to_json(
        "fob_reference.json",
        orient="records"
    )

# Scenario Analytics

**Publication review note.** Analytics helpers below can be called after generating each desired scenario. Automatic loading of three nonexistent or inconsistently named scenario folders was removed.


In [ ]:
def load_scenario_results(scenario_name, input_root="results"):
    base = Path(input_root) / scenario_name
    return {
        "df_baseline": pd.read_parquet(base / "baseline.parquet"),
        "df_scenario": pd.read_parquet(base / "scenario.parquet"),
        "df_compare": pd.read_parquet(base / "compare.parquet"),
        "config": json.load(open(base / "config.json"))
    }


df_fob_ref = (
    df_bltr_multivar
    .loc[df_bltr_multivar["fob_per_ton_seamodes"].notna()]
    .groupby(["origin_isocode","destination_isocode","product_code"], as_index=False)
    .agg(fob_per_ton_ref=("fob_per_ton_seamodes","median"))
)
def prepare_analytics_df(results_dict, df_fob_ref):
    dfs = []

    for case, res in results_dict.items():
        df = res["df_compare"].copy()
        df["case"] = case

        df = df.drop(columns=["fob_per_ton_ref"], errors="ignore").merge(
            df_fob_ref,
            on=["origin_isocode","destination_isocode","product_code"],
            how="left"
        )

        df["tc_fob_ratio_baseline"] = (
            df["transport_cost_per_ton_pred_baseline"] / df["fob_per_ton_ref"]
        )

        df["tc_fob_ratio_stress"] = (
            df["transport_cost_per_ton_pred_stress"] / df["fob_per_ton_ref"]
        )

        df["delta_tc_fob_pp"] = (
            df["tc_fob_ratio_stress"] - df["tc_fob_ratio_baseline"]
        ) * 100

        df["trade_index"] = (
            df["tons_seamode_pred_stress"] /
            df["tons_seamode_pred_baseline"]
        ) * 100

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

# 8. EXPORT FOR DASHBOARD

In [ ]:
Path("dashboard_data").mkdir(exist_ok=True)
df_compare.to_parquet("dashboard_data/scenario_compare.parquet")

print("Simulation completed successfully.")

# Misc. Custom Case config generator

### Optional EU ETS configuration generator

The helper is defined here without automatically replacing the configuration used above. Call `generate_eu_ets_json(results["df_baseline"])` explicitly to save a candidate configuration.


In [ ]:
import json
from pathlib import Path
import pandas as pd

# ============================================================
# PARAMETERS
# ============================================================

EU_ISOCODES = [
    "AUT","BEL","BGR","HRV","CYP","CZE","DNK","EST","FIN","FRA",
    "DEU","GRC","HUN","IRL","ITA","LVA","LTU","LUX","MLT","NLD",
    "POL","PRT","ROU","SVK","SVN","ESP","SWE"
]

PRODUCT_CODES = ["1001", "1005"]
SHARE_THRESHOLD = 0.85
START_YEAR = 2025
END_YEAR = 2030

OUTPUT_PATH = Path("generated_configs/eu_ets.json")


# ============================================================
# SELECT AFFECTED CORRIDORS
# ============================================================

def select_eu_ets_corridors(
    df_baseline: pd.DataFrame,
    eligibility_rule,
    product_codes,
    share_threshold=0.85,
    start_year=2025,
    end_year=2030,
    min_corridors=1
):
    df = df_baseline.copy()
    df = df[(df["year"] >= start_year) & (df["year"] <= end_year)]
    df = df[eligibility_rule(df)]
    df = df[df["product_code"].isin(product_codes)]

    if df.empty:
        raise ValueError("No corridors match eligibility rule.")

    corr = (
        df.groupby(["origin_isocode", "destination_isocode"], as_index=False)
          .agg(baseline_tons=("tons_seamode_pred", "sum"))
          .sort_values("baseline_tons", ascending=False)
    )

    corr["share"] = corr["baseline_tons"] / corr["baseline_tons"].sum()
    corr["cum_share"] = corr["share"].cumsum()

    selected = corr[corr["cum_share"] <= share_threshold]

    if selected.shape[0] < min_corridors:
        selected = corr.head(min_corridors)

    return [
        [o, d]  # Convert to JSON-safe list
        for o, d in zip(selected["origin_isocode"], selected["destination_isocode"])
    ]


# ============================================================
# EU ETS RULE
# ============================================================

def eu_ets_rule(df):
    return (
        df["origin_isocode"].isin(EU_ISOCODES) |
        df["destination_isocode"].isin(EU_ISOCODES)
    )


# ============================================================
# BUILD SCENARIO CONFIG
# ============================================================

def build_eu_ets_config(df_baseline: pd.DataFrame):

    affected_corridors = select_eu_ets_corridors(
        df_baseline=df_baseline,
        eligibility_rule=eu_ets_rule,
        product_codes=PRODUCT_CODES,
        share_threshold=SHARE_THRESHOLD,
        start_year=START_YEAR,
        end_year=END_YEAR
    )

    scenario_config = {
        "scenario_name": "EU ETS Maritime Regulation",

        "affected_corridors": affected_corridors,

        "affected_product_codes": PRODUCT_CODES,

        "transport_shocks": {
            "freight_proxy_fbx": 0.12,  # ETS cost pass-through
            "connectivity_lsbci": 0.00,
            "distance_proxy": 0.00,
            "bdi_proxy": 0.00
        },

        "country_shocks": {
            "production_by_origin_country": {},
            "demand_by_destination_country": {}
        },

        "apply_demand_to_calorific_proxy": False
    }

    return scenario_config


# ============================================================
# SAVE FUNCTION
# ============================================================

def save_config(config: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"✅ EU ETS config saved to: {path}")


# ============================================================
# MAIN EXECUTION
# ============================================================

def generate_eu_ets_json(df_baseline: pd.DataFrame):
    config = build_eu_ets_config(df_baseline)
    save_config(config, OUTPUT_PATH)


In [ ]:
# from scenario_generator import build_eu_ets_scenario, save_scenario_config

# scenario_config_eu_ets = build_eu_ets_scenario(results["df_baseline"])

# save_scenario_config(scenario_config_eu_ets, "eu_ets")